In [1]:
import sys
print(sys.version)
print(sys.executable)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
C:\Users\User\anaconda3\python.exe


In [ ]:
%pip install flask gunicorn

In [4]:
import flask 
print("Flask version:", flask.__version__)

Flask version: 3.1.2


C:\Users\User\AppData\Local\Temp\ipykernel_23580\2685938378.py:2: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print("Flask version:", flask.__version__)


In [5]:
import os 
os.makedirs("Student_Task_Manager/templates", exist_ok=True)
os.makedirs("Student_Task_Manager/static", exist_ok=True)
os.chdir("Student_Task_Manager")
print(os.getcwd())

C:\Users\User\Student_Task_Manager


In [ ]:

%%writefile app.py

from flask import Flask, render_template, request, redirect, url_for
import sqlite3
from pathlib import Path

app = Flask(__name__)

DATABASE = Path(__file__).with_name("tasks.db")


# Database connection
def get_db():
    conn = sqlite3.connect(DATABASE)
    conn.row_factory = sqlite3.Row
    return conn


# Create database table
def create_table():
    conn = get_db()

    conn.execute("""
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            description TEXT,
            completed INTEGER DEFAULT 0
        )
    """)

    conn.commit()
    conn.close()


# Home page - display all tasks
@app.route("/")
def index():

    conn = get_db()

    tasks = conn.execute(
        "SELECT * FROM tasks ORDER BY id DESC"
    ).fetchall()

    conn.close()

    return render_template("index.html", tasks=tasks)


# Add a new task
@app.route("/add", methods=["POST"])
def add_task():

    title = request.form.get("title", "").strip()
    description = request.form.get("description", "").strip()

    if title:
        conn = get_db()

        conn.execute(
            "INSERT INTO tasks (title, description) VALUES (?, ?)",
            (title, description)
        )

        conn.commit()
        conn.close()

    return redirect(url_for("index"))


# Edit task
@app.route("/edit/<int:task_id>", methods=["GET", "POST"])
def edit_task(task_id):

    conn = get_db()

    task = conn.execute(
        "SELECT * FROM tasks WHERE id = ?",
        (task_id,)
    ).fetchone()

    if task is None:
        conn.close()
        return "Task not found", 404

    if request.method == "POST":

        title = request.form.get("title", "").strip()
        description = request.form.get("description", "").strip()

        if title:
            conn.execute(
                """UPDATE tasks
                   SET title = ?, description = ?
                   WHERE id = ?""",
                (title, description, task_id)
            )

            conn.commit()
            conn.close()

            return redirect(url_for("index"))

    conn.close()

    return render_template("edit.html", task=task)


# Mark task as completed
@app.route("/complete/<int:task_id>", methods=["POST"])
def complete_task(task_id):

    conn = get_db()

    conn.execute(
        """UPDATE tasks
           SET completed = CASE
               WHEN completed = 0 THEN 1
               ELSE 0
           END
           WHERE id = ?""",
        (task_id,)
    )

    conn.commit()
    conn.close()

    return redirect(url_for("index"))


# Delete task
@app.route("/delete/<int:task_id>", methods=["POST"])
def delete_task(task_id):

    conn = get_db()

    conn.execute(
        "DELETE FROM tasks WHERE id = ?",
        (task_id,)
    )

    conn.commit()
    conn.close()

    return redirect(url_for("index"))


if __name__ == "__main__":

    create_table()

    app.run(
        host="127.0.0.1",
        port=5000,
        debug=False
    )
# Initialize database for both local and AWS deployment
create_table()


if __name__ == "__main__":

    app.run(
        host="127.0.0.1",
        port=5000,
        debug=False
    )

In [7]:

%%writefile templates/index.html

<!DOCTYPE html>
<html>
<head>
    <title>Student Task Manager</title>

    <link rel="stylesheet"
          href="{{ url_for('static', filename='style.css') }}">
</head>

<body>

<div class="container">

    <h1>Student Task Manager</h1>
    <p class="subtitle">Manage your daily activities easily</p>

    <div class="form-box">

        <h2>Add New Task</h2>

        <form action="/add" method="POST">

            <input type="text"
                   name="title"
                   placeholder="Enter task title"
                   required>

            <textarea name="description"
                      placeholder="Enter task description"></textarea>

            <button type="submit">Add Task</button>

        </form>

    </div>

    <h2>My Tasks</h2>

    {% for task in tasks %}

    <div class="task-card">

        <h3>
            {% if task['completed'] %}
                <span class="done">{{ task['title'] }}</span>
            {% else %}
                {{ task['title'] }}
            {% endif %}
        </h3>

        <p>{{ task['description'] }}</p>

        <p>
            Status:
            {% if task['completed'] %}
                <span class="done">Completed</span>
            {% else %}
                <span class="pending">Pending</span>
            {% endif %}
        </p>

        <div class="actions">

            <form action="/complete/{{ task['id'] }}"
                  method="POST">
                <button class="complete" type="submit">
                    Toggle Status
                </button>
            </form>

            <a class="edit"
               href="/edit/{{ task['id'] }}">
                Edit
            </a>

            <form action="/delete/{{ task['id'] }}"
                  method="POST">
                <button class="delete" type="submit">
                    Delete
                </button>
            </form>

        </div>

    </div>

    {% else %}

        <p>No tasks available. Add your first task!</p>

    {% endfor %}

</div>

</body>
</html>


Writing templates/index.html


In [8]:

%%writefile templates/edit.html

<!DOCTYPE html>
<html>
<head>
    <title>Edit Task</title>

    <link rel="stylesheet"
          href="{{ url_for('static', filename='style.css') }}">
</head>

<body>

<div class="container">

    <h1>Edit Task</h1>

    <div class="form-box">

        <form method="POST">

            <input type="text"
                   name="title"
                   value="{{ task['title'] }}"
                   required>

            <textarea name="description">{{ task['description'] }}</textarea>

            <button type="submit">Update Task</button>

        </form>

        <a href="/">Back to Home</a>

    </div>

</div>

</body>
</html>


Writing templates/edit.html


In [9]:

%%writefile static/style.css

body {
    font-family: Arial, sans-serif;
    background: #f1f5f9;
    margin: 0;
    padding: 30px;
}

.container {
    max-width: 850px;
    margin: auto;
}

h1 {
    text-align: center;
    color: #1e3a8a;
}

.subtitle {
    text-align: center;
    color: #64748b;
}

.form-box {
    background: white;
    padding: 25px;
    border-radius: 10px;
    margin: 25px 0;
    box-shadow: 0 3px 10px #ddd;
}

input, textarea {
    width: 95%;
    padding: 12px;
    margin: 10px 0;
    border: 1px solid #cbd5e1;
    border-radius: 6px;
}

textarea {
    min-height: 70px;
}

button, .edit {
    padding: 10px 16px;
    border: none;
    border-radius: 5px;
    cursor: pointer;
    text-decoration: none;
    color: white;
    background: #2563eb;
}

.task-card {
    background: white;
    padding: 20px;
    margin: 15px 0;
    border-radius: 8px;
    box-shadow: 0 2px 7px #ddd;
}

.actions {
    display: flex;
    gap: 10px;
    flex-wrap: wrap;
    align-items: center;
}

.complete {
    background: #16a34a;
}

.delete {
    background: #dc2626;
}

.edit {
    background: #f59e0b;
}

.done {
    color: #16a34a;
    text-decoration: line-through;
}

.pending {
    color: #ea580c;
}


Writing static/style.css


In [10]:

%%writefile requirements.txt

Flask>=3.0,<4.0
gunicorn>=22.0,<24.0


Writing requirements.txt


In [ ]:
!python app.py

In [ ]:
import os

for root, dirs, files in os.walk("."):
    print(root, files)

In [ ]:
%%writefile .gitignore

tasks.db
__pycache__/
.venv/
.env